In [19]:
import pandas as pd
import os

# 1. Chargement des données Silver
# Assurez-vous d'être dans le bon chemin relatif par rapport au notebook
silver_security_path = "../../data/silver/securite_data_silver_clean.csv"

print(f"Chargement du fichier : {silver_security_path}")
df_sec = pd.read_csv(silver_security_path, sep=";")
display(df_sec.head())

Chargement du fichier : ../../data/silver/securite_data_silver_clean.csv


,code_departement,code_commune,code_insee_commune,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux,source_dataset,source_dataset_timestamp,date_refresh
0,69,1,69001,2016,Violences physiques intrafamiliales,Victime,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
1,69,1,69001,2016,Violences physiques hors cadre familial,Victime,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
2,69,1,69001,2016,Violences sexuelles,Victime,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
3,69,1,69001,2016,Vols avec armes,Infraction,0.000000,0.000000,diff,358,2016-01-01,163,2016-01-01,0.000000,0.000000,securite_commune,2026-03-23 14:13:09.597346,2026-03-23
4,69,1,69001,2016,Vols violents sans arme,Infraction,1.581633,0.317097,ndiff,358,2016-01-01,163,2016-01-01,1.581633,0.317097,securite_commune,2026-03-23 14:13:09.597346,2026-03-23


In [20]:
# 2. Pivotement (Long vers Wide)
# L'objectif est d'utiliser `taux_pour_mille` comme valeur indicative de la criminalité normalisée à la population.

df_gold_sec = df_sec.pivot_table(
    index=["code_departement", "code_commune", "code_insee_commune", "annee", "insee_pop"],
    columns="indicateur",
    values="taux_pour_mille",
    aggfunc="sum"
).reset_index()

# 3. Nettoyage des noms de colonnes
# Suppression des majuscules, espaces et caractères spéciaux pour les entêtes du ML
df_gold_sec.columns.name = None  # retirer le nom d'index 'indicateur'

def clean_col_name(c):
    return str(c).strip().lower().replace(" ", "_").replace("'", "_").replace("-", "_")

df_gold_sec.columns = [clean_col_name(col) for col in df_gold_sec.columns]

print(f"Format final après pivot : {df_gold_sec.shape}")
display(df_gold_sec.head())

Format final après pivot : (2475, 20)


,code_departement,code_commune,code_insee_commune,annee,insee_pop,cambriolages_de_logement,destructions_et_dégradations_volontaires,escroqueries_et_fraudes_aux_moyens_de_paiement,trafic_de_stupéfiants,usage_de_stupéfiants,usage_de_stupéfiants_(afd),violences_physiques_hors_cadre_familial,violences_physiques_intrafamiliales,violences_sexuelles,vols_avec_armes,vols_d_accessoires_sur_véhicules,vols_dans_les_véhicules,vols_de_véhicule,vols_sans_violence_contre_des_personnes,vols_violents_sans_arme
0,69,1,69001,2016,358,6.780636,3.344787,2.651935,0.0,1.736679,0.0,0.0,0.000000,0.0,0.0,0.0,1.912863,0.0,2.043030,0.317097
1,69,1,69001,2017,369,5.271140,3.568340,2.793816,0.0,1.282986,0.0,0.0,0.000000,0.0,0.0,0.0,1.593679,0.0,2.189426,0.311020
2,69,1,69001,2018,379,4.767782,3.672004,3.163971,0.0,0.972048,0.0,0.0,0.000000,0.0,0.0,0.0,1.540558,0.0,1.895187,0.253448
3,69,1,69001,2019,391,5.390536,3.380606,3.256909,0.0,0.934652,0.0,0.0,1.434160,0.0,0.0,0.0,1.702461,0.0,2.087375,0.237294
4,69,1,69001,2020,393,4.271015,2.838793,3.876028,0.0,0.000000,0.0,0.0,1.685316,0.0,0.0,0.0,1.326652,0.0,1.714131,0.000000


In [21]:
# 4. Sauvegarde dans la couche Gold

gold_output_path = "../../data/gold/dim_securite.csv"
os.makedirs(os.path.dirname(gold_output_path), exist_ok=True)

df_gold_sec.to_csv(gold_output_path, sep=";", index=False)

print(f"✅ Jeu de données Gold des indicateurs de sécurité sauvegardé à : {gold_output_path}")

✅ Jeu de données Gold des indicateurs de sécurité sauvegardé à : ../../data/gold/dim_securite.csv
